## Data Exploration using batch inspection
- This means we explore 12 csv files all together to get a bird's eye view

In [3]:
from pathlib import Path
import pandas as pd

In [4]:
def inspect_all_csvs(raw_data_dir = "/Users/Priyanka/Documents/Internship-Internmo/Zomato_BI_Project/data/raw"):
    """
    Inspects all CSV files in the given directory and prints their shape and column names.

    Parameters:
    raw_data_dir (str): The path to the directory containing the CSV files.
    """
    raw_data_path = Path(raw_data_dir)
    
    for csv_file in raw_data_path.glob("*.csv"):
        df = pd.read_csv(csv_file)
        print(f"\nFile: {csv_file.name}")
        print(f"    - Shape: {df.shape}")
        print(f"    - Columns: {df.columns.tolist()}")
        print(f"    - Duplicates: {df.duplicated().sum()}")
        print("-" * 40)

        #Missing values check
        missing_values = df.isnull().sum()
        if missing_values.any():
            print(f"    - Missing Values:")
            for col, count in missing_values[missing_values > 0].items():
                print(f"      - {col}: {count} ({count/len(df)*100:.1f}%)")
        else:
            print("    - No missing values found.")

In [5]:
#Execute the function to inspect all CSV files in the specified directory
inspect_all_csvs()


File: customers.csv
    - Shape: (12180, 13)
    - Columns: ['CustomerID', 'Name', 'Age', 'Gender', 'Phone', 'Email', 'City', 'State', 'Pincode', 'RegistrationDate', 'Membership', 'TotalOrders', 'PreferredCuisine']
    - Duplicates: 180
----------------------------------------
    - Missing Values:
      - Age: 170 (1.4%)
      - Gender: 122 (1.0%)
      - Phone: 144 (1.2%)
      - Email: 332 (2.7%)
      - State: 186 (1.5%)
      - Membership: 120 (1.0%)
      - PreferredCuisine: 245 (2.0%)

File: delivery_partners.csv
    - Shape: (2000, 10)
    - Columns: ['DeliveryPartnerID', 'Name', 'Age', 'Gender', 'VehicleType', 'JoiningDate', 'City', 'Rating', 'CompletedDeliveries', 'AverageDeliveryTime']
    - Duplicates: 0
----------------------------------------
    - Missing Values:
      - Rating: 34 (1.7%)

File: orders.csv
    - Shape: (20705, 15)
    - Columns: ['OrderID', 'CustomerID', 'RestaurantID', 'DeliveryPartnerID', 'OrderDate', 'OrderTime', 'DeliveryTimeMinutes', 'FoodCost', 'D

## Deep dive Profile Function
- This function helps to inspect the important csv files further by performing a deep-dive numeric and categorical breakdown before cleaning. 

In [10]:
def profile_dataset(df,name="dataset",categorical_cols =None):
    """
    Profiles the given dataset and prints summary statistics.

    Parameters:
    df (pd.DataFrame): The DataFrame to profile.
    name (str): The name of the dataset for display purposes.
    categorical_cols (list): List of categorical columns to profile.
    """
    print ("=" * 50)
    print(f"DETAILED PROFILE OF {name.upper()}:")
    print ("=" * 50)

    #1. Non-null counts and data types
    print("\n---Data Types and Data Completeness---")
    info_df = pd.DataFrame(
        {
            "DataType": df.dtypes,
            "Non-Null Count": df.notnull().sum(),
            "Missing Count": df.isnull().sum(),
            "Missing Percentage": (df.isnull().sum() / len(df) * 100).round(2),

        }
    )
    print(info_df)



    #2. Numeric Summary Statistic
    num_cols = df.select_dtypes(include=['number']).columns
    if len(num_cols) > 0:
        print("\n---Numeric Summary Statistics---")
        stats = df[num_cols].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
        stats["IQR"] = stats["75%"] - stats["25%"]
        stats["Outliers"] = ((df[num_cols] < (stats["25%"] - 1.5 * stats["IQR"])) | (df[num_cols] > (stats["75%"] + 1.5 * stats["IQR"]))).sum()
        print(stats.round(2))



    #3. Categorical Summary Statistic
    if categorical_cols:
        print("\n---Categorical Summary Statistics---")
        for col in categorical_cols:
            if col in df.columns:
                print(f"\nValue Counts for '{col}':")
                print(df[col].value_counts(dropna=False))
            else:
                print(f"\nColumn: {col} not found in the dataset.")
    print("\n" + "=" * 50 + "\n")

In [11]:
#Deep dive into the orders dataset
orders_df = pd.read_csv(
    "/Users/Priyanka/Documents/Internship-Internmo/Zomato_BI_Project/data/raw/orders.csv"
)
profile_dataset(
    orders_df,
    name="Orders",
    categorical_cols=["OrderStatus", "PaymentMethod", "CouponCode"],
)

DETAILED PROFILE OF ORDERS:

---Data Types and Data Completeness---
                    DataType  Non-Null Count  Missing Count  \
OrderID                int64           20705              0   
CustomerID             int64           20705              0   
RestaurantID           int64           20705              0   
DeliveryPartnerID      int64           20705              0   
OrderDate                str           20705              0   
OrderTime                str           20705              0   
DeliveryTimeMinutes    int64           20705              0   
FoodCost               int64           20705              0   
DeliveryFee            int64           20705              0   
Discount             float64           20705              0   
CouponCode               str            7284          13421   
GST                  float64           20705              0   
FinalAmount          float64           20378            327   
OrderStatus              str           20705      

In [13]:
#Deep dive into the restaurants dataset
restaurants_df = pd.read_csv(
    "/Users/Priyanka/Documents/Internship-Internmo/Zomato_BI_Project/data/raw/restaurants.csv"
)
profile_dataset(
    restaurants_df,
    name="Restaurants",
    categorical_cols=["Cuisine", "OpeningTime","ClosingTime","City", "OwnerName", "RestaurantType"]
)

DETAILED PROFILE OF RESTAURANTS:

---Data Types and Data Completeness---
               DataType  Non-Null Count  Missing Count  Missing Percentage
RestaurantID      int64            1200              0                0.00
RestaurantName      str            1200              0                0.00
Cuisine             str            1188             12                1.00
City                str            1200              0                0.00
Area                str            1200              0                0.00
OpeningTime         str            1200              0                0.00
ClosingTime         str            1200              0                0.00
Rating          float64            1178             22                1.83
AverageCost       int64            1200              0                0.00
OwnerName           str            1200              0                0.00
RestaurantType      str            1200              0                0.00
Latitude        float64    

In [14]:
#Deep dive into the customers dataset
customers_df = pd.read_csv(
    "/Users/Priyanka/Documents/Internship-Internmo/Zomato_BI_Project/data/raw/customers.csv"
)
profile_dataset(
    customers_df,
    name="Customers",
    categorical_cols=["Gender", "City","Membership","PreferredCuisine"]
)

DETAILED PROFILE OF CUSTOMERS:

---Data Types and Data Completeness---
                 DataType  Non-Null Count  Missing Count  Missing Percentage
CustomerID          int64           12180              0                0.00
Name                  str           12180              0                0.00
Age               float64           12010            170                1.40
Gender                str           12058            122                1.00
Phone                 str           12036            144                1.18
Email                 str           11848            332                2.73
City                  str           12180              0                0.00
State                 str           11994            186                1.53
Pincode               str           12180              0                0.00
RegistrationDate      str           12180              0                0.00
Membership            str           12060            120                0.99
Total

In [15]:
#deep dive into the delivery_partners dataset
delivery_partners_df = pd.read_csv(
    "/Users/Priyanka/Documents/Internship-Internmo/Zomato_BI_Project/data/raw/delivery_partners.csv"
)
profile_dataset(
    delivery_partners_df,
    name="Delivery Partners",
    categorical_cols=["VehicleType", "City", "Gender"]
)

DETAILED PROFILE OF DELIVERY PARTNERS:

---Data Types and Data Completeness---
                    DataType  Non-Null Count  Missing Count  \
DeliveryPartnerID      int64            2000              0   
Name                     str            2000              0   
Age                    int64            2000              0   
Gender                   str            2000              0   
VehicleType              str            2000              0   
JoiningDate              str            2000              0   
City                     str            2000              0   
Rating               float64            1966             34   
CompletedDeliveries    int64            2000              0   
AverageDeliveryTime  float64            2000              0   

                     Missing Percentage  
DeliveryPartnerID                   0.0  
Name                                0.0  
Age                                 0.0  
Gender                              0.0  
VehicleType      

In [16]:
#Deep profiling of the menu dataset
menu_df = pd.read_csv(
    "/Users/Priyanka/Documents/Internship-Internmo/Zomato_BI_Project/data/raw/menu.csv"
)
profile_dataset(
    menu_df,
    name="Menu",
    categorical_cols=["Availability", "Category"]
)

DETAILED PROFILE OF MENU:

---Data Types and Data Completeness---
                DataType  Non-Null Count  Missing Count  Missing Percentage
FoodItemID         int64            8997              0                0.00
RestaurantID       int64            8997              0                0.00
FoodName             str            8997              0                0.00
Category             str            8997              0                0.00
Price              int64            8997              0                0.00
PreparationTime    int64            8997              0                0.00
Calories         float64            8865            132                1.47
Availability         str            8997              0                0.00

---Numeric Summary Statistics---
                  count     mean      std    min     25%     50%     75%  \
FoodItemID       8997.0  4499.00  2597.35    1.0  2250.0  4499.0  6748.0   
RestaurantID     8997.0   601.39   346.24    1.0   302.0   602.0

In [17]:
#Deep dive into the traffic dataset
traffic_df = pd.read_csv(
    "/Users/Priyanka/Documents/Internship-Internmo/Zomato_BI_Project/data/raw/traffic.csv"
)
profile_dataset(
    traffic_df,
    name="Traffic",
    categorical_cols=["City", "TrafficLevel"]
)

DETAILED PROFILE OF TRAFFIC:

---Data Types and Data Completeness---
             DataType  Non-Null Count  Missing Count  Missing Percentage
TrafficID       int64           18335              0                 0.0
City              str           18335              0                 0.0
Date              str           18335              0                 0.0
Time              str           18335              0                 0.0
TrafficLevel      str           18188            147                 0.8
AverageSpeed  float64           18335              0                 0.0

---Numeric Summary Statistics---
                count     mean      std   min     25%     50%      75%  \
TrafficID     18335.0  9168.00  5293.00   1.0  4584.5  9168.0  13751.5   
AverageSpeed  18335.0    23.48    14.34 -47.7    13.9    23.0     32.4   

                  max     IQR  Outliers  
TrafficID     18335.0  9167.0         0  
AverageSpeed    119.9    18.5       299  

---Categorical Summary Statistics---

In [18]:
#Deep dive into the weather dataset
weather_df = pd.read_csv(
    "/Users/Priyanka/Documents/Internship-Internmo/Zomato_BI_Project/data/raw/weather.csv"
)
profile_dataset(
    weather_df,
    name="Weather",
    categorical_cols=["City", "WeatherCondition"]
)

DETAILED PROFILE OF WEATHER:

---Data Types and Data Completeness---
                 DataType  Non-Null Count  Missing Count  Missing Percentage
WeatherID           int64           18264              0                0.00
City                  str           18173             91                0.50
Date                  str           18264              0                0.00
Temperature       float64           18264              0                0.00
Rainfall          float64           17995            269                1.47
Humidity            int64           18264              0                0.00
WeatherCondition      str           18264              0                0.00

---Numeric Summary Statistics---
               count     mean      std   min      25%     50%       75%  \
WeatherID    18264.0  9132.50  5272.51   1.0  4566.75  9132.5  13698.25   
Temperature  18264.0    26.80     7.30   5.0    22.00    27.2     31.50   
Rainfall     17995.0    35.85    77.45   0.0     0.00   